# Group 5 – PayShield Fintech
## Member 1: Data Engineering & Preprocessing

This notebook contains **only the Data Engineer tasks** for the Credit Card Fraud Detection mini-project.

### Responsibilities covered
1. Load and document the raw dataset
2. Validate the dataset schema
3. Inspect data types
4. Check and handle missing values
5. Check and handle duplicate records
6. Validate transaction IDs
7. Validate invalid values
8. Clean categorical/text fields
9. Convert date fields
10. Engineer reusable features
11. Handle transaction amount appropriately
12. Separate features and target
13. Perform a stratified train/test split
14. Build leakage-safe preprocessing
15. Scale numerical variables using training data only
16. Save processed train/test datasets
17. Generate a data-quality audit log
18. Provide a reproducible preprocessing function for `src/preprocessing.py`

> **Important:** EDA interpretation, model building, model evaluation, dashboard work, and business recommendations belong to other project tasks and are intentionally not performed here.


In [ ]:
# ============================================
# 1. IMPORT LIBRARIES AND CONFIGURATION
# ============================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Libraries imported successfully.")


## 2. Project paths

The project is designed to run from VS Code/GitHub without using a personal Windows path.

Expected structure:

```text
project/
│
├── data/
│   ├── raw/
│   │   └── credit_card_fraud.csv
│   └── processed/
│
├── notebooks/
│   └── Member1_Data_Engineer.ipynb
│
├── src/
│   └── preprocessing.py
│
├── requirements.txt
└── README.md
```


In [ ]:
# ============================================
# 2. PROJECT PATHS
# ============================================

# This notebook should be inside the "notebooks" folder.
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SRC_DIR = PROJECT_ROOT / "src"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DIR / "credit_card_fraud.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data path:", DATA_PATH)
print("Processed data folder:", PROCESSED_DIR)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Place credit_card_fraud.csv inside data/raw/."
    )


In [ ]:
# ============================================
# 3. LOAD RAW DATA
# ============================================

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Shape:", df.shape)

display(df.head())


## 3. Dataset acquisition documentation

The raw dataset is treated as the source dataset. The raw file is kept unchanged in `data/raw/`.

The Data Engineer should not overwrite the raw dataset during cleaning or feature engineering.


In [ ]:
# ============================================
# 4. BASIC DATASET INSPECTION
# ============================================

print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDataset information:")
df.info()

print("\nDescriptive statistics:")
display(df.describe(include="all").T)


In [ ]:
# ============================================
# 5. SCHEMA VALIDATION
# ============================================

expected_columns = {
    "trans_date_trans_time",
    "merchant",
    "category",
    "amt",
    "city",
    "state",
    "lat",
    "long",
    "city_pop",
    "job",
    "dob",
    "trans_num",
    "merch_lat",
    "merch_long",
    "is_fraud"
}

actual_columns = set(df.columns)

missing_columns = expected_columns - actual_columns
unexpected_columns = actual_columns - expected_columns

print("Missing columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

assert not missing_columns, f"Missing required columns: {missing_columns}"

print("\nSchema validation passed.")


In [ ]:
# ============================================
# 6. DATA TYPE CONVERSION
# ============================================

df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"],
    errors="coerce"
)

df["dob"] = pd.to_datetime(
    df["dob"],
    errors="coerce"
)

print("Date columns converted.")
display(df[["trans_date_trans_time", "dob"]].head())

print("\nMissing values created by invalid date conversion:")
display(df[["trans_date_trans_time", "dob"]].isnull().sum())


In [ ]:
# ============================================
# 7. MISSING-VALUE CHECK
# ============================================

missing_counts = df.isnull().sum()

print("Total missing values:", missing_counts.sum())

if missing_counts.sum() == 0:
    print("No missing values found.")
else:
    display(missing_counts[missing_counts > 0])


### Missing-value handling decision

If the dataset contains no missing values, no artificial imputation is required.

If missing values are discovered, the appropriate treatment must be documented rather than silently deleting rows.


In [ ]:
# ============================================
# 8. DUPLICATE CHECKS
# ============================================

duplicate_rows = df.duplicated().sum()
duplicate_transaction_ids = df["trans_num"].duplicated().sum()

print("Duplicate rows:", duplicate_rows)
print("Duplicate transaction IDs:", duplicate_transaction_ids)

if duplicate_rows > 0:
    print("\nDuplicate rows found. Removing exact duplicate rows.")
    df = df.drop_duplicates().reset_index(drop=True)

if duplicate_transaction_ids > 0:
    print("\nWarning: duplicate transaction IDs remain.")
else:
    print("\nTransaction ID uniqueness check passed.")

print("Shape after duplicate handling:", df.shape)


In [ ]:
# ============================================
# 9. INVALID-VALUE VALIDATION
# ============================================

invalid_checks = {
    "negative_amounts": (df["amt"] < 0).sum(),
    "invalid_latitude": ((df["lat"] < -90) | (df["lat"] > 90)).sum(),
    "invalid_longitude": ((df["long"] < -180) | (df["long"] > 180)).sum(),
    "invalid_merchant_latitude": ((df["merch_lat"] < -90) | (df["merch_lat"] > 90)).sum(),
    "invalid_merchant_longitude": ((df["merch_long"] < -180) | (df["merch_long"] > 180)).sum(),
    "negative_city_population": (df["city_pop"] < 0).sum(),
    "invalid_target_values": (~df["is_fraud"].isin([0, 1])).sum()
}

invalid_results = pd.Series(invalid_checks, name="count")

display(invalid_results.to_frame())

assert invalid_checks["invalid_target_values"] == 0,     "Invalid values found in is_fraud."

assert invalid_checks["invalid_latitude"] == 0,     "Invalid customer latitude values found."

assert invalid_checks["invalid_longitude"] == 0,     "Invalid customer longitude values found."

assert invalid_checks["invalid_merchant_latitude"] == 0,     "Invalid merchant latitude values found."

assert invalid_checks["invalid_merchant_longitude"] == 0,     "Invalid merchant longitude values found."

print("Target and geographic validation passed.")


### Invalid-value treatment

The code validates values that have clear domain constraints.

It does **not** automatically delete unusual transaction amounts because extreme transactions can contain useful fraud information. Outlier detection and fraud-related interpretation should not be confused with invalid-record removal.


In [ ]:
# ============================================
# 10. TEXT/CATEGORICAL CLEANING
# ============================================

text_columns = ["merchant", "category", "city", "job"]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

df["category"] = df["category"].str.lower()
df["state"] = df["state"].astype("string").str.strip().str.upper()

print("Text cleaning completed.")

for col in ["category", "state"]:
    print(f"\nSample values in {col}:")
    print(df[col].dropna().unique()[:10])


In [ ]:
# ============================================
# 11. TARGET VALIDATION
# ============================================

target_counts = df["is_fraud"].value_counts().sort_index()

print("Target distribution:")
display(target_counts)

print("\nTarget proportions:")
display(df["is_fraud"].value_counts(normalize=True).sort_index())

assert set(df["is_fraud"].unique()).issubset({0, 1})

print("\nTarget validation passed.")


## 12. Feature engineering

The following features are created once and used consistently:

- `age`
- `hour`
- `day_of_week`
- `month`
- `log_amt`
- `distance_km`

The transaction ID is removed because it is an identifier rather than a predictive feature.


In [ ]:
# ============================================
# 12. FEATURE ENGINEERING
# ============================================

# Age
df["age"] = (
    (df["trans_date_trans_time"] - df["dob"]).dt.days / 365.25
).astype("Int64")

# Time features
df["hour"] = df["trans_date_trans_time"].dt.hour
df["day_of_week"] = df["trans_date_trans_time"].dt.dayofweek
df["month"] = df["trans_date_trans_time"].dt.month

# Log transaction amount
df["log_amt"] = np.log1p(df["amt"])

# Haversine geographical distance in kilometres
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine_distance(
    df["lat"],
    df["long"],
    df["merch_lat"],
    df["merch_long"]
)

# Remove identifier and DOB after creating age
df = df.drop(columns=["trans_num", "dob"])

print("Feature engineering completed.")
display(df.head())


## 13. Outlier strategy

The raw transaction amount is strongly right-skewed, so `log_amt` is created.

We do **not** blindly delete high-value transactions because an extreme transaction can be a genuine transaction or a useful fraud signal.

Most importantly, preprocessing parameters that are learned from the data must not be calculated using the test set.


In [ ]:
# ============================================
# 13. PREPARE FEATURES AND TARGET
# ============================================

target = "is_fraud"

X = df.drop(columns=[target])
y = df[target].astype(int)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

assert target not in X.columns
assert len(X) == len(y)

print("Target successfully separated from features.")


In [ ]:
# ============================================
# 14. TRAIN/TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nFraud percentage:")
print("Train:", round(y_train.mean() * 100, 4), "%")
print("Test :", round(y_test.mean() * 100, 4), "%")

print("\nStratification check:")
print("Difference:", abs(y_train.mean() - y_test.mean()))


### Why the split happens before learned preprocessing

The train/test split is performed before fitting the scaler or encoder.

This prevents information from the test set from being used to learn preprocessing parameters.

This is an important **data-leakage prevention** step.


In [ ]:
# ============================================
# 15. DEFINE PREPROCESSING COLUMNS
# ============================================

# Columns that should be treated as numerical
numeric_features = [
    "amt",
    "log_amt",
    "lat",
    "long",
    "city_pop",
    "merch_lat",
    "merch_long",
    "age",
    "hour",
    "day_of_week",
    "month",
    "distance_km"
]

# Lower-cardinality categorical feature retained for preprocessing
categorical_features = [
    "category"
]

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

missing_numeric = set(numeric_features) - set(X_train.columns)
missing_categorical = set(categorical_features) - set(X_train.columns)

assert not missing_numeric, f"Missing numerical features: {missing_numeric}"
assert not missing_categorical, f"Missing categorical features: {missing_categorical}"


In [ ]:
# ============================================
# 16. LEAKAGE-SAFE PREPROCESSING PIPELINE
# ============================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first",
                sparse_output=False
            ),
            categorical_features
        )
    ],
    remainder="drop"
)

# IMPORTANT:
# Fit ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using the already-fitted preprocessing
X_test_processed = preprocessor.transform(X_test)

print("Training processed shape:", X_train_processed.shape)
print("Testing processed shape :", X_test_processed.shape)
print("Leakage-safe preprocessing completed.")


In [ ]:
# ============================================
# 17. CREATE PROCESSED FEATURE NAMES
# ============================================

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Processed feature count:", len(feature_names))

display(X_train_processed.head())


In [ ]:
# ============================================
# 18. FINAL PROCESSED DATA VALIDATION
# ============================================

print("X_train shape:", X_train_processed.shape)
print("X_test shape :", X_test_processed.shape)

print("\nMissing values in X_train:",
      X_train_processed.isnull().sum().sum())

print("Missing values in X_test:",
      X_test_processed.isnull().sum().sum())

print("\nTarget values:")
print("y_train:", y_train.value_counts().to_dict())
print("y_test :", y_test.value_counts().to_dict())

assert X_train_processed.isnull().sum().sum() == 0
assert X_test_processed.isnull().sum().sum() == 0

assert len(X_train_processed) == len(y_train)
assert len(X_test_processed) == len(y_test)

print("\nFinal processed-data validation passed.")


In [ ]:
# ============================================
# 19. SAVE TRAIN/TEST DATA
# ============================================

train_final = X_train_processed.copy()
train_final["is_fraud"] = y_train.values

test_final = X_test_processed.copy()
test_final["is_fraud"] = y_test.values

X_train_path = PROCESSED_DIR / "X_train.csv"
X_test_path = PROCESSED_DIR / "X_test.csv"
train_path = PROCESSED_DIR / "train_processed.csv"
test_path = PROCESSED_DIR / "test_processed.csv"

X_train_processed.to_csv(X_train_path, index=False)
X_test_processed.to_csv(X_test_path, index=False)

train_final.to_csv(train_path, index=False)
test_final.to_csv(test_path, index=False)

print("Saved files:")
print(X_train_path)
print(X_test_path)
print(train_path)
print(test_path)


In [ ]:
# ============================================
# 20. DATA-QUALITY AUDIT LOG
# ============================================

quality_log = pd.DataFrame({
    "Check": [
        "Expected schema",
        "Missing values",
        "Duplicate rows before cleaning",
        "Duplicate transaction IDs",
        "Negative transaction amounts",
        "Invalid customer latitude",
        "Invalid customer longitude",
        "Invalid merchant latitude",
        "Invalid merchant longitude",
        "Negative city population",
        "Invalid target values"
    ],
    "Result": [
        "Passed" if not missing_columns else str(missing_columns),
        int(missing_counts.sum()),
        int(duplicate_rows),
        int(duplicate_transaction_ids),
        int(invalid_checks["negative_amounts"]),
        int(invalid_checks["invalid_latitude"]),
        int(invalid_checks["invalid_longitude"]),
        int(invalid_checks["invalid_merchant_latitude"]),
        int(invalid_checks["invalid_merchant_longitude"]),
        int(invalid_checks["negative_city_population"]),
        int(invalid_checks["invalid_target_values"])
    ]
})

display(quality_log)

quality_log_path = PROCESSED_DIR / "data_quality_log.csv"
quality_log.to_csv(quality_log_path, index=False)

print("Audit log saved to:", quality_log_path)


## 21. Reproducibility check

This section verifies that the final train/test datasets have:

- matching feature columns
- no missing processed values
- separated target
- reproducible random state
- consistent preprocessing
- no use of the test set when fitting the preprocessing transformer


In [ ]:
# ============================================
# 21. REPRODUCIBILITY / FINAL CHECKS
# ============================================

assert list(X_train_processed.columns) == list(X_test_processed.columns)

assert "is_fraud" not in X_train_processed.columns
assert "is_fraud" not in X_test_processed.columns

assert X_train_processed.shape[1] == X_test_processed.shape[1]

assert len(X_train_processed) == len(y_train)
assert len(X_test_processed) == len(y_test)

print("============================================")
print("MEMBER 1 DATA ENGINEERING CHECKS PASSED")
print("============================================")
print("Schema validation       : PASS")
print("Missing-value handling  : PASS")
print("Duplicate handling      : PASS")
print("Invalid-value checks    : PASS")
print("Feature engineering     : PASS")
print("Train/test split        : PASS")
print("Stratification          : PASS")
print("Leakage-safe preprocessing: PASS")
print("Processed data saved    : PASS")
print("Quality log saved       : PASS")


# Member 1 deliverables

After running this notebook, the Data Engineer should have:

```text
data/
├── raw/
│   └── credit_card_fraud.csv
│
└── processed/
    ├── X_train.csv
    ├── X_test.csv
    ├── train_processed.csv
    ├── test_processed.csv
    └── data_quality_log.csv
```

The reusable functions should also be copied into:

```text
src/
└── preprocessing.py
```

### Important

Do not perform SMOTE, undersampling, oversampling, model training, model evaluation, dashboard creation, or business recommendations in this Member 1 notebook.

If resampling is required later, it must be performed **after the train/test split and only on the training data**, preferably inside the modelling pipeline.
